Colab is making it easier than ever to integrate powerful Generative AI capabilities into your projects. We are launching public preview for a simple and intuitive Python library (google.colab.ai) to access state-of-the-art language models directly within Colab environments. All users have free access to most popular LLMs, while paid users have access to a wider selection of models. This means users can spend less time on configuration and set up and more time bringing their ideas to life. With just a few lines of code, you can now perform a variety of tasks:
- Generate text
- Translate languages
- Write creative content
- Categorize text

Happy Coding!


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb)

In [9]:
def extract_invoice_fields(text):
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system="""You are an expert at extracting structured data from
commercial invoices for customs declarations in Latin America.
Be precise. If a field is not found, write NOT FOUND.""",
        messages=[
            {
                "role": "user",
                "content": f"""Extract these customs-relevant fields from the invoice text below.
Return ONLY a structured list, nothing else.

Fields:
- Shipper name and address
- Consignee name and address
- Invoice number
- Invoice date
- HS Code(s)
- Description of goods
- Declared value and currency
- Country of origin
- Net weight
- Gross weight

Invoice text:
{text[:4000]}

⚠️ HUMAN REVIEW REQUIRED: These results are AI-assisted.
Verify all fields before submitting to customs authorities."""
            }
        ]
    )
    return response.content[0].text

# Test it — will return NOT FOUND for most fields since this is a plan doc, not an invoice
# That is expected and correct behavior
result = extract_invoice_fields(text)
print(result)

# In Colab — after uploading the invoice
text = extract_text_from_pdf("/content/synthetic_invoice_001.pdf")
result = extract_invoice_fields(text)
print(result)

## Extracted Customs Fields — Invoice INV-2026-00847

---

- **Shipper name and address:**
GlobalTech Imports LLC, 1847 NW 79th Avenue, Suite 310, Miami, Florida 33126, USA

- **Consignee name and address:**
TechSolutions S.A. de C.V., Blvd. de los Héroes #2341, Colonia Escalón, San Salvador, El Salvador, C.A. CP 01101

- **Invoice number:**
INV-2026-00847

- **Invoice date:**
April 18, 2026

- **HS Code(s):**
8471.30.00 | 8471.60.10 | 8471.60.20 | 8528.52.00 | 8504.40.10 | 8536.69.90

- **Description of goods:**
1. Laptop Computer, Intel Core i7-1355U, 16GB RAM, 512GB SSD, 15.6" Display (Qty: 25)
2. External USB Keyboard, Wireless, Spanish Layout QWERTY-LA (Qty: 50)
3. Optical Mouse, Wireless, USB Receiver, 1600 DPI (Qty: 50)
4. LED Monitor 24", Full HD 1920x1080, HDMI/VGA, 75Hz (Qty: 15)
5. UPS Uninterruptible Power Supply, 1000VA/600W, 8 Outlets (Qty: 20)
6. Network Switch 24-Port, Gigabit Ethernet, Unmanaged (Qty: 10)

- **Declared value and currency:**
USD $42,515.00 (Total Invoic

In [8]:
import sys

if 'google.colab' in sys.modules:
  %pip install anthropic

import anthropic
from pydantic import BaseModel
from typing import Optional
from google.colab import userdata

class Message(BaseModel):
    role: str
    content: str

class Conversation(BaseModel):
    messages: list[Message] = []

    def add_message(self, role: str, content: str):
        msg = Message(role=role, content=content)
        self.messages.append(msg)
        return msg

    def to_api_format(self, ):
        return [{"role": m.role, "content": m.content} for m in self.messages]

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

convo = Conversation()
system_prompt = "You are an expert in customs and logistics for Central America. Answer clearly and concisely."

print("Chat with your AI (type 'quit' to stop)\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == "quit":
        break

    convo.add_message("user", user_input)

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system=system_prompt,
        messages=convo.to_api_format()
    )

    reply = response.content[0].text
    convo.add_message("assistant", reply)
    print(f"\nAI: {reply}\n")

print(f"\nConversation had {len(convo.messages)} messages total.")

Chat with your AI (type 'quit' to stop)

You: quit

Conversation had 0 messages total.


In [5]:
!pip install anthropic pdfplumber --quiet

import anthropic
import pdfplumber
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

# Test with any PDF — even a random one to start
# Upload a file to Colab first using the file panel on the left

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

# Once you have a PDF uploaded, test it:
text = extract_text_from_pdf("/content/AI_Business_Plan_Final.pdf")
print(text[:2000])

# Send extracted text to Claude and ask it to identify key fields
# This simulates what we'll do with a real invoice

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    system="You are an expert at extracting structured data from trade documents.",
    messages=[
        {
            "role": "user",
            "content": f"""Extract the following fields from this document.
If a field is not found, write 'NOT FOUND'.

Fields to extract:
- Document title
- Author/Organization
- Main goal or purpose
- Key dates mentioned
- Any numerical targets mentioned

Document text:
{text[:3000]}

Return the results as a clean structured list."""
        }
    ]
)

print(response.content[0].text)

AI Business Plan
Spanish-Language Trade Document Intelligence
Systems Engineer · DHL Express El Salvador
Version 5 — Final · April 2026
Incorporating: Original Plan + ChatGPT + Gemini evaluations
GOAL 1 — PRODUCT GOAL 2 — CAREER
$1,000–$3,000+ MRR within 12 months through Senior IT/AI engineering role — every task builds portfolio
AI-powered trade document extraction tool evidence simultaneously1. Who I Am & My Real Constraints
Systems Engineer at DHL Express, El Salvador — customs project implementation and IT infrastructure. Some
coding experience (HTML, basic scripts), now learning Python and AI development.
WORK SCHEDULE ENERGY REALITY
15 hrs/week — weekends only. This is the ceiling, not the Mentally draining DHL week. 5–6 hrs deep work/day max.
floor. Saturday AM is peak.
REST RULE FINANCIAL
One full rest weekend per month. Planned in advance. 12+ months runway. No immediate income pressure.
Non-negotiable.
CAREER OPEN NETWORK
Open to senior IT/AI role offers at any point during 

In [ ]:
from pydantic import BaseModel
from typing import Optional

class Message(BaseModel):
    role: str
    content: str
    tokens: Optional[int] = None

class Conversation(BaseModel):
    messages: list[Message] = []
    total_tokens: int = 0

    def add_message(self, role: str, content: str):
        self.messages.append(Message(role=role, content=content))

# Test it
msg = Message(role="user", content="Hello AI world!")
print(msg)

convo = Conversation()
convo.add_message("user", "Hello!")
convo.add_message("assistant", "Hi there!")
print(convo)

role='user' content='Hello AI world!' tokens=None
messages=[Message(role='user', content='Hello!', tokens=None), Message(role='assistant', content='Hi there!', tokens=None)] total_tokens=0


In [ ]:
import anthropic
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    system="You are a helpful assistant for someone learning AI engineering.",
    messages=[
        {"role": "user", "content": "Hello Claude! I just wrote my first Pydantic model. What should I learn next?"}
    ]
)

print(message.content[0].text)

Congratulations on writing your first Pydantic model! That's a great foundation. Here are some logical next steps to build on what you've learned:

## Immediate Next Steps:
1. **Pydantic Validators** - Learn custom validation with `@validator` or `@field_validator` (v2)
2. **Type Annotations** - Explore advanced types like `Optional`, `List`, `Dict`, `Union`
3. **Model Configuration** - Understanding `Config` class or `model_config` (v2)

## Practical Applications:
4. **API Integration** - Use your models with **FastAPI** for request/response validation
5. **Data Serialization** - Learn `.dict()`, `.json()`, and parsing from JSON/dict
6. **Error Handling** - Working with `ValidationError`

## Intermediate Concepts:
7. **Model Inheritance** - Creating base models and extending them
8. **Field Types** - `Field()` for additional constraints and metadata
9. **Nested Models** - Models containing other models

## What was your first model for? 
Knowing your use case would help me suggest the

In [ ]:
!pip install anthropic pydantic python-dotenv httpx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.5/627.5 kB 9.2 MB/s eta 0:00:00


In [ ]:
# @title List available models
from google.colab import ai

ai.list_models()

['google/gemini-2.5-flash', 'google/gemini-2.5-flash-lite']

Choosing a Model
The model names give you a hint about their capabilities and intended use:

Pro: These are the most capable models, ideal for complex reasoning, creative tasks, and detailed analysis.

Flash: These models are optimized for high speed and efficiency, making them great for summarization, chat applications, and tasks requiring rapid responses.

Gemma: These are lightweight, open-weight models suitable for a variety of text generation tasks and are great for experimentation.

In [ ]:
# @title Simple batch generation example
# Only text-to-text input/output is supported
from google.colab import ai

response = ai.generate_text("What is the capital of France?")
print(response)

The capital of France is **Paris**.


For longer text generations, you can stream the response. This displays the output token by token as it's generated, rather than waiting for the entire response to complete. This provides a more interactive and responsive experience. To enable this, simply set stream=True.

In [ ]:
# @title Simple streaming example
from google.colab import ai

stream = ai.generate_text("Tell me a short story.", stream=True)
for text in stream:
  print(text, end='')

In [ ]:
#@title Text formatting setup
#code is not necessary for colab.ai, but is useful in fomatting text chunks
import sys

class LineWrapper:
    def __init__(self, max_length=80):
        self.max_length = max_length
        self.current_line_length = 0

    def print(self, text_chunk):
        i = 0
        n = len(text_chunk)
        while i < n:
            start_index = i
            while i < n and text_chunk[i] not in ' \n': # Find end of word
                i += 1
            current_word = text_chunk[start_index:i]

            delimiter = ""
            if i < n: # If not end of chunk, we found a delimiter
                delimiter = text_chunk[i]
                i += 1 # Consume delimiter

            if current_word:
                needs_leading_space = (self.current_line_length > 0)

                # Case 1: Word itself is too long for a line (must be broken)
                if len(current_word) > self.max_length:
                    if needs_leading_space: # Newline if current line has content
                        sys.stdout.write('\n')
                        self.current_line_length = 0
                    for char_val in current_word: # Break the long word
                        if self.current_line_length >= self.max_length:
                            sys.stdout.write('\n')
                            self.current_line_length = 0
                        sys.stdout.write(char_val)
                        self.current_line_length += 1
                # Case 2: Word doesn't fit on current line (print on new line)
                elif self.current_line_length + (1 if needs_leading_space else 0) + len(current_word) > self.max_length:
                    sys.stdout.write('\n')
                    sys.stdout.write(current_word)
                    self.current_line_length = len(current_word)
                # Case 3: Word fits on current line
                else:
                    if needs_leading_space:
                        # Define punctuation that should not have a leading space
                        # when they form an entire "word" (token) following another word.
                        no_leading_space_punctuation = {
                            ",", ".", ";", ":", "!", "?",        # Standard sentence punctuation
                            ")", "]", "}",                     # Closing brackets
                            "'s", "'S", "'re", "'RE", "'ve", "'VE", # Common contractions
                            "'m", "'M", "'ll", "'LL", "'d", "'D",
                            "n't", "N'T",
                            "...", "…"                          # Ellipses
                        }
                        if current_word not in no_leading_space_punctuation:
                            sys.stdout.write(' ')
                            self.current_line_length += 1
                    sys.stdout.write(current_word)
                    self.current_line_length += len(current_word)

            if delimiter == '\n':
                sys.stdout.write('\n')
                self.current_line_length = 0
            elif delimiter == ' ':
                # If line is full and a space delimiter arrives, it implies a wrap.
                if self.current_line_length >= self.max_length:
                    sys.stdout.write('\n')
                    self.current_line_length = 0

        sys.stdout.flush()


In [ ]:
# @title Formatted streaming example
from google.colab import ai

wrapper = LineWrapper()
for chunk in ai.generate_text('Give me a long winded description about the evolution of the Roman Empire.', model_name='google/gemini-2.0-flash', stream=True):
  wrapper.print(chunk)